# RoB2 RAG Chatbot

Same RAG pipeline as `grade_rag.ipynb` — only the source URLs and system prompt differ.

**Source:** [Risk of Bias 2.0 Tool](https://www.riskofbias.info/welcome/rob-2-0-tool)

**RoB2 domains covered:**
1. Bias arising from the randomization process
2. Bias due to deviations from intended interventions
3. Bias due to missing outcome data
4. Bias in measurement of the outcome
5. Bias in selection of the reported result

In [5]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from dotenv import load_dotenv
load_dotenv()

from rag_core import get_pdf_paths, load_or_build_index, rag_answer, build_llm

## 1. Configuration

In [6]:
ROB2_SYSTEM_PROMPT = (
    "You are an expert assistant specialising in the Cochrane Risk of Bias tool 2.0 (RoB2). "
    "RoB2 assesses risk of bias in randomized trials across five domains: "
    "(1) randomization process, (2) deviations from intended interventions, "
    "(3) missing outcome data, (4) outcome measurement, (5) selection of the reported result. "
    "Answer the user's question based ONLY on the retrieved RoB2 content below. "
    "Do not fabricate or speculate beyond what is in the context. "
    "If the context does not contain enough information, say so explicitly. "
    "Use precise RoB2 terminology (signalling questions, bias judgments: Low/Some concerns/High, etc.)."
)

ROB2_PDF_DIR = "data/rob2_pdfs"
PARSED_ROB2_PDF_DIR = "data/parsed_rob2"
CACHE_DIR = ".faiss_cache/rob2"

rob2_pdf_paths = get_pdf_paths(ROB2_PDF_DIR)
print(f"Found {len(rob2_pdf_paths)} RoB2 PDFs")

Found 3 RoB2 PDFs


## 2. Build / Load Index

First run scrapes all RoB2 pages and saves a local FAISS cache.  
Subsequent runs load from cache instantly.

> **Note:** some subpages on riskofbias.info may not be JavaScript-rendered.  
> If a page returns empty text, increase `wait_time` below.

In [7]:
import shutil
shutil.rmtree(".faiss_cache/rob2", ignore_errors=True)                                               
print("Cache cleared.")

Cache cleared.


In [8]:
index = load_or_build_index(
    pdf_paths=rob2_pdf_paths,
    cache_path=CACHE_DIR,
    parsed_pdf_cache_dir=PARSED_ROB2_PDF_DIR,
)
llm = build_llm()
print("Ready.")

Building index from scratch...
  Parsing PDF: data/rob2_pdfs/1.pdf
    -> 103 documents
  Parsing PDF: data/rob2_pdfs/2.pdf
    -> 32 documents
  Parsing PDF: data/rob2_pdfs/3.pdf
    -> 17 documents
Total documents: 152


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index saved to: .faiss_cache/rob2
Ready.


## 3. Inspect Scraped Content

Verify the index contains meaningful RoB2 text before evaluation.

In [9]:
# Quick sanity check: retrieve top chunks for a known RoB2 concept
probe_query = "signalling questions randomization domain"
docs = index.similarity_search(probe_query, k=3)
for i, doc in enumerate(docs, 1):
    print(f"--- Chunk {i} (source: {doc.metadata['source']}) ---")
    print(doc.page_content[:400])
    print()

--- Chunk 1 (source: 1.pdf) ---
Signalling questions and criteria for judging risk of bias
Signalling questions for this domain are provided in Box 11. Criteria for reaching risk-of-bias judgements are given in Table 13, and an algorithm for implementing these is provided in Table 14 and 

--- Chunk 2 (source: 1.pdf) ---
Signalling questions
Inclusion of signalling questions within each domain of bias is a key feature of RoB 2. Signalling questions aim to elicit information relevant to an assessment of risk of bias. They seek to be reasonably factual in nature. Responses to these questions feed into algorithms we have developed to guide users of the tool to judgements about the risk of bias.
The response options f

--- Chunk 3 (source: 2.pdf) ---
Signalling questions Elaboration 1b.1 Were all the individual participants identified and recruited (if appropriate) before randomization of clusters?
Answer 'Yes' if:
(1) all participants were identified and recruited before the clusters were

## 4. Ask a Question

In [10]:
query = "What are the five domains assessed in RoB2?"
answer, contexts = rag_answer(query, index, llm, k=5, system_prompt=ROB2_SYSTEM_PROMPT)
print("Answer:\n", answer)
print("\n--- Retrieved contexts ---")
for i, ctx in enumerate(contexts, 1):
    print(f"[{i}] {ctx[:200]}...")

Answer:
 The five domains assessed in RoB2 are:
1. randomization process
2. deviations from intended interventions
3. missing outcome data
4. outcome measurement
5. selection of the reported result

--- Retrieved contexts ---
[1] Presentation of risk-of-bias assessments
We suggest that RoB 2 assessments are presented as follows. More work is required in this area.
• For full transparency of the process, review authors may wish...
[2] ... multiple eligible outcome measurements (e.g. scales, definitions, time points) within the outcome domain?
A particular outcome domain (i.e. a true state or endpoint of interest) may be measured in...
[3] Domain-level judgements about risk of bias
RoB 2 is conceived hierarchically: responses to signalling questions elicit what happened and provide the basis for domain-level judgements about the risk of...
[4] ... multiple eligible analyses of the data?
A particular outcome measurement may be analysed in multiple ways. Examples include: unadjusted and ad

## 5. Sanity-Check Questions

In [11]:
sanity_questions = [
    "In my risk of bias assessment, can I add an extra domain corresponding to the quality of statistical results presentation?",
    "Do the different risk of bias domains have different implications in the overall result classification?",
    "I have five domains classified as 'Low risk of bias' and one domain classified as 'High risk of bias'. What is the overall risk of bias judgement?",
    "Is it adequate to stop risk of bias assessment once one of the domains is judged at high risk of bias?",
    "In my systematic review, I am assessing an educational intervention. Can I assume upfront that it is not possible to implement allocation concealment?",
    # "What makes an outcome measurement blinded vs unblinded in RoB2?",
    # "What is selective outcome reporting and which domain covers it?",
    # "When would you rate a trial as 'High' risk of bias for Domain 2?",
    # "What is the overall RoB2 judgment if one domain is rated High?",
    # "How does RoB2 differ from the original Cochrane Risk of Bias tool (RoB1)?",
]

for q in sanity_questions:
    ans, _ = rag_answer(q, index, llm, k=5, system_prompt=ROB2_SYSTEM_PROMPT)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}")
    print()

Q: In my risk of bias assessment, can I add an extra domain corresponding to the quality of statistical results presentation?
A: The provided context does not contain enough information to answer whether an extra domain corresponding to the quality of statistical results presentation can be added to the RoB2 assessment. The context explicitly states that RoB2 assesses risk of bias across five domains: (1) randomization proces

Q: Do the different risk of bias domains have different implications in the overall result classification?
A: No, the different risk of bias domains do not have different implications in the overall result classification. The context states that "Domain-level judgements about risk of bias should have the same implication for each of the six domains with respect to concern about the impact of bias on the tru

Q: I have five domains classified as 'Low risk of bias' and one domain classified as 'High risk of bias'. What is the overall risk of bias judgement?
A: Base

In [12]:
docs = index.similarity_search("risk of bias domain signalling questions", k=5)                      
for i, doc in enumerate(docs, 1):                                                                    
      print(f"--- Chunk {i} ---")                                                                      
      print(f"Source: {doc.metadata['source']}")                                                       
      print(doc.page_content[:300])                                                                    
      print()  

--- Chunk 1 ---
Source: 1.pdf
Signalling questions and criteria for judging risk of bias
Signalling questions for this domain are provided in Box 11. Criteria for reaching risk-of-bias judgements are given in Table 13, and an algorithm for implementing these is provided in Table 14 and 

--- Chunk 2 ---
Source: 1.pdf
Signalling questions
Inclusion of signalling questions within each domain of bias is a key feature of RoB 2. Signalling questions aim to elicit information relevant to an assessment of risk of bias. They seek to be reasonably factual in nature. Responses to these questions feed into algorithms we ha

--- Chunk 3 ---
Source: 1.pdf
Some concerns
The study is judged to raise some concerns in at least one domain for this result, but not to be at high risk of bias for any domain.

--- Chunk 4 ---
Source: 1.pdf
High risk of bias
The study is judged to be at high risk of bias in at least one domain for this result.

--- Chunk 5 ---
Source: 1.pdf
Domain-level judgements about risk

## 6. Convenience Functions (for evaluation notebooks)

In [15]:
def ask_rob2(question: str) -> str:
    answer, _ = rag_answer(question, index, llm, system_prompt=ROB2_SYSTEM_PROMPT)
    return answer

def get_rob2_contexts(question: str, k: int = 5) -> list[str]:
    _, contexts = rag_answer(question, index, llm, k=k, system_prompt=ROB2_SYSTEM_PROMPT)
    return contexts

## 7. Optional Gradio UI

In [17]:
import gradio as gr

def gradio_fn(question, history):
    answer, _ = rag_answer(question, index, llm, system_prompt=ROB2_SYSTEM_PROMPT)
    return answer

demo = gr.ChatInterface(
    fn=gradio_fn,
    title="RoB2 Assistant",
    description="Ask questions about the Cochrane Risk of Bias 2.0 tool. Answers are grounded in official RoB2 documentation.",
)
# Uncomment to launch:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
